[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Train YOLO11 Object Detection on a Custom Dataset

---

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/ultralytics/ultralytics)

YOLO11 builds on the advancements introduced in YOLOv9 and YOLOv10 earlier this year, incorporating improved architectural designs, enhanced feature extraction techniques, and optimized training methods.

YOLO11m achieves a higher mean mAP score on the COCO dataset while using 22% fewer parameters than YOLOv8m, making it computationally lighter without sacrificing performance.

YOLOv11 is available in 5 different sizes, ranging from `2.6M` to `56.9M` parameters, and capable of achieving from `39.5` to `54.7` mAP on the COCO dataset.

## Setup

### Configure API keys

To fine-tune YOLO11, you need to provide your Roboflow API key. Follow these steps:

- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

### Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [2]:
!nvidia-smi

Wed Mar 11 13:05:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [16]:
import os
HOME = os.getcwd()
print(HOME)

/content


## Install YOLO11 via Ultralytics

In [4]:
%pip install "ultralytics<=8.3.40" supervision roboflow
# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()

Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.8/112.6 GB disk)


## Inference with model pre-trained on COCO dataset

### CLI

**NOTE:** CLI requires no customization or Python code. You can simply run all tasks from the terminal with the yolo command.

In [5]:
!yolo task=detect mode=predict model=yolo11n.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg' save=True

Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 238 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

Found https://media.roboflow.com/notebooks/examples/dog.jpeg locally at dog.jpeg
image 1/1 /content/dog.jpeg: 640x384 2 persons, 1 car, 1 dog, 1 handbag, 47.5ms
Speed: 6.9ms preprocess, 47.5ms inference, 193.3ms postprocess per image at shape (1, 3, 640, 384)
Results saved to runs/detect/predict3
💡 Learn more at https://docs.ultralytics.com/modes/predict


In [17]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="zEDXk03GPE1VtWJS0gtx")
project = rf.workspace("aziizahs-workspace").project("tugas-akhir-new-dsv89")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


In [18]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt

In [19]:
# 3️⃣ Upload YOLOv11 weights
# Pastikan kamu sudah upload yolov11n.pt ke folder /content/
YOLO11_WEIGHTS = "/content/yolo11m.pt"

# 4️⃣ Load YOLOv11 model
model = YOLO(YOLO11_WEIGHTS)
DATA_YAML = "/content/Tugas-Akhir-New-1/data.yaml"  # ganti sesuai lokasi data.yaml

In [20]:
# 6️⃣ Output folder
PROJECT = "TugasAkhir-YOLO"
NAME = "yolov11_training"
os.makedirs(f"{PROJECT}/{NAME}", exist_ok=True)

In [21]:
model.train(
    data=DATA_YAML,
    epochs=80,           # training lebih lama
    imgsz=640,
    batch=16,            # lebih stabil untuk GPU T4
    lr0=0.002,           # learning rate sedikit lebih tinggi untuk percepatan
    optimizer="AdamW",
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    augment=True
    # patience=15
)

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/content/yolo11m.pt, data=/content/Tugas-Akhir-New-1/data.yaml, epochs=80, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=TugasAkhir-YOLO, name=yolov11_training, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=Fa

train: Scanning /content/Tugas-Akhir-New-1/train/labels.cache... 22 images, 20 backgrounds, 0 corrupt: 100%|██████████| 22/22 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



Argument(s) 'quality_lower' are not valid for transform ImageCompression
val: Scanning /content/Tugas-Akhir-New-1/valid/labels.cache... 7 images, 5 backgrounds, 0 corrupt: 100%|██████████| 7/7 [00:00<?, ?it/s]


Plotting labels to TugasAkhir-YOLO/yolov11_training/labels.jpg... 
optimizer: AdamW(lr=0.002, momentum=0.937) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.0005), 112 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to TugasAkhir-YOLO/yolov11_training
Starting training for 80 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all          7          2      0.236        0.5      0.194      0.133



80 epochs completed in 0.080 hours.
Optimizer stripped from TugasAkhir-YOLO/yolov11_training/weights/last.pt, 40.5MB
Optimizer stripped from TugasAkhir-YOLO/yolov11_training/weights/best.pt, 40.5MB

Validating TugasAkhir-YOLO/yolov11_training/weights/best.pt...
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 303 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:03<00:00,  3.78s/it]


                   all          7          2     0.0403          1      0.152      0.122
Speed: 0.1ms preprocess, 536.4ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to TugasAkhir-YOLO/yolov11_training


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f70e24a9220>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

**NOTE:** The obtained `result` object stores information about the location, classes, and confidence levels of the detected objects.

In [22]:
metrics = model.val()
metrics_dict = metrics.results_dict

precision = metrics_dict['metrics/precision(B)']
recall = metrics_dict['metrics/recall(B)']
map50 = metrics_dict['metrics/mAP50(B)']
map50_95 = metrics_dict['metrics/mAP50-95(B)']
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5-0.95: {map50_95:.4f}")
print(f"F1-score: {f1_score:.4f}")

Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 303 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs


val: Scanning /content/Tugas-Akhir-New-1/valid/labels.cache... 7 images, 5 backgrounds, 0 corrupt: 100%|██████████| 7/7 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]


                   all          7          2     0.0406          1      0.152      0.123
Speed: 0.2ms preprocess, 55.5ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to TugasAkhir-YOLO/yolov11_training
Precision: 0.0406
Recall: 1.0000
mAP@0.5: 0.1515
mAP@0.5-0.95: 0.1234
F1-score: 0.0780


In [23]:
from google.colab import files

# Path ke model yang sudah di-train
model_path = "/content/TugasAkhir-YOLO/yolov11_training/weights/best.pt"

# Download ke PC
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>